# Users Feature engineering

We will create the columns that are intersing for our users

In [1]:
import pandas as pd

In [3]:
sessions = pd.read_csv('data/sessions.csv')
users = pd.read_csv('data/users.csv')
flights = pd.read_csv('data/flights.csv')
hotels = pd.read_csv('data/hotels.csv')

## Columns directly from users table

In [4]:
users.columns

Index(['user_id', 'birthdate', 'gender', 'married', 'has_children',
       'home_country', 'home_city', 'home_airport', 'home_airport_lat',
       'home_airport_lon', 'sign_up_date'],
      dtype='str')

directly took columns from user table

In [5]:
user_features = users[['user_id', 'gender', 'married', 'has_children']].copy()

## Tenure

In [6]:
user_features['tenure'] = (pd.Timestamp.today() - pd.to_datetime(users['sign_up_date'])).dt.days # feature column

Idea: From this column create group of new customers.

## Overcarrier

In [7]:
flights

,trip_id,origin_airport,destination,destination_airport,seats,return_flight_booked,departure_time,return_time,checked_bags,trip_airline,destination_airport_lat,destination_airport_lon,base_fare_usd
0,363535-ae2567b185da4e3994607ce71f98a96b,CLT,phoenix,PHX,1,False,2023-01-13 15:00:00.000000,NaN,0,Alaska Airlines,33.535,-112.383,251.87
1,549482-f5e7931dd7b6460b90b89ea0aaabfc78,YVR,san diego,SAN,1,True,2023-07-25 11:00:00.000000,2023-07-31 11:00:00.000000,1,American Airlines,32.699,-117.215,369.26
2,585745-7f13ca3cb64441e08ba34a3020e7ff7b,LBB,new york,JFK,1,True,2023-07-24 07:00:00.000000,2023-07-29 07:00:00.000000,0,Allegiant Air,40.640,-73.779,407.51
3,596153-eb736cf7fe6f4aaf8c1638e399111ced,YKZ,calgary,YYC,1,True,2023-07-25 15:00:00.000000,2023-07-27 15:00:00.000000,1,WestJet,51.114,-114.020,449.58
4,636526-80cabddfe58c406d9ba90c77f85c66ff,LGA,los angeles,LAX,1,True,2023-07-23 13:00:00.000000,2023-07-26 13:00:00.000000,1,Air New Zealand,33.942,-118.408,691.85
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13188,544449-e96a99d225be4ca2862bfa16d02f94d9,LGA,chicago,MDW,2,True,2023-03-30 08:00:00.000000,2023-04-01 08:00:00.000000,0,AirTran Airways,41.786,-87.752,389.69
13189,566735-978b549aa0804742883e1a10e57d6790,BOS,indianapolis,IND,1,True,2023-04-02 09:00:00.000000,2023-04-06 09:00:00.000000,1,Allegiant Air,39.717,-86.294,175.15
13190,580611-b8fc699133a8430a935067cd85f68549,JFK,montreal,YUL,1,True,2023-03-31 11:00:00.000000,2023-04-05 11:00:00.000000,0,Royal Jordanian,45.517,-73.417,97.40
13191,588386-233db0f627cf4aaaaeaa9a632daa5ec4,CLE,philadelphia,PHL,1,True,2023-04-04 08:00:00.000000,2023-04-06 08:00:00.000000,2,American Airlines,39.872,-75.241,106.00


In [10]:
flights_w_users = pd.merge(flights, sessions[['trip_id','user_id']], on='trip_id', how='left')

filter out valid trips and calculate average bags per seat

In [11]:
cancelled_trip_ids = sessions[sessions['cancellation']]['trip_id']


In [12]:
seats_avg = flights_w_users[~flights_w_users['trip_id'].isin(cancelled_trip_ids)].groupby('user_id')['seats'].mean()

In [13]:
checked_bags_avg = flights_w_users[~flights_w_users['trip_id'].isin(cancelled_trip_ids)].groupby('user_id')['checked_bags'].mean()

In [65]:
bags_per_seat = checked_bags_avg/seats_avg

In [66]:
overcarrier = bags_per_seat > 2  # feature column

## Frequent travelers

In [44]:
total_trips  = sessions.groupby('user_id')['trip_id'].nunique() # feature column

In [52]:
number_of_cancellations = sessions.groupby('user_id')['cancellation'].sum()

In [55]:
cancellation_rate  = number_of_cancellations/total_trips  # feature column